# Servidor Único de Inferencia en Google Colab (1 Clic)

Levanta el endpoint único (`/inferencia`) en GPU de Google Colab con:
- **M1 + OCR multi-ángulo** (corrección de inclinación/perspectiva)
- **M2**: Segmentación de vehículos (`moto`, `auto`, `minibus`, `camion`, `bus`)
- **M3**: Conteo de ejes (`1, 2, 3, 4, 5, 6+` ejes desde cualquier ángulo)
- **M4**: Segmentación y conteo de pasajeros visibles dentro del vehículo (sin rostros)

Al ejecutar la Celda 2 te imprimirá una URL `https://....trycloudflare.com`.
Copia esa URL y pégala directamente en tu panel web (`https://peaje-ia-web.onrender.com` -> pestaña **Estado** -> **INFERENCE_URL**).

In [ ]:
!rm -rf /content/proyecto_caseta_IA
!git clone https://github.com/yeixd/proyecto_caseta_IA.git /content/proyecto_caseta_IA
%cd /content/proyecto_caseta_IA/inferencia
!pip install -q fastapi uvicorn[standard] python-multipart pydantic ultralytics easyocr opencv-python-headless python-dotenv
!python diagnostico_entorno.py

In [ ]:
# (Opcional) Si ya tienes tus pesos entrenados .pt en Google Drive, descomenta las 2 líneas siguientes:
# from google.colab import drive
# drive.mount('/content/drive')

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

import os, subprocess, time, re

os.environ['MODEL_M1_PLATES_PATH'] = '/content/drive/MyDrive/proyecto_peaje/models/m1_placas.pt'
os.environ['MODEL_M2_VEHICLE_SEG_PATH'] = '/content/drive/MyDrive/proyecto_peaje/models/m2_vehiculos_seg.pt'
os.environ['MODEL_M3_WHEEL_POSE_PATH'] = '/content/drive/MyDrive/proyecto_peaje/models/m3_ruedas_pose.pt'
os.environ['MODEL_M4_PASSENGERS_SEG_PATH'] = '/content/drive/MyDrive/proyecto_peaje/models/m4_pasajeros_seg.pt'
# Sin clave obligatoria para pruebas rápidas (o definir la misma en la pestaña Estado)
os.environ['INFERENCE_API_KEY'] = ''

server = subprocess.Popen(['uvicorn', 'main:app', '--host', '0.0.0.0', '--port', '8000'])
print('Cargando modelos YOLO + EasyOCR en GPU (espera ~15 segundos)...')
time.sleep(12)

tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)
for line in tunnel.stdout:
    m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if m:
        print('\n=================================================================')
        print('COPIA ESTA URL Y PÉGALA EN ESTADO -> INFERENCE_URL EN TU WEB:')
        print(m.group(0))
        print('=================================================================\n')
        break

print('Servidor activo. Mantén esta celda corriendo mientras pruebas.')
server.wait()